In [ ]:
# second_level_features code

# Enhanced Frequency Domain CNN with Dual-Domain Feature Fusion (EfficientNet-B0)
# Key Innovation: Second-level FFT on last conv layer activation maps

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm
import os
import warnings
import gc
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""
    
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== Step 2: FIXED FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """Convert spatial domain images to frequency domain using FFT"""
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    
    mean_val = freq_magnitude_log.mean()
    std_val = freq_magnitude_log.std()
    std_val = torch.clamp(std_val, min=1e-5)
    
    freq_magnitude_normalized = (freq_magnitude_log - mean_val) / std_val
    freq_magnitude_normalized = torch.clamp(freq_magnitude_normalized, -10, 10)
    
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    freq_features = torch.nan_to_num(freq_features, nan=0.0, posinf=10.0, neginf=-10.0)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    freq_magnitude = torch.exp(torch.clamp(freq_magnitude, -10, 10))
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

def extract_frequency_features(activation_map):
    """
    Extract second-level frequency features from activation maps
    This captures hidden patterns in the frequency domain
    """
    freq_complex = fft.fft2(activation_map, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    eps = 1e-6
    
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    
    b, c, h, w = freq_magnitude_log.shape
    freq_magnitude_norm = freq_magnitude_log.view(b, c, -1)
    
    mean_val = freq_magnitude_norm.mean(dim=2, keepdim=True)
    std_val = freq_magnitude_norm.std(dim=2, keepdim=True)
    std_val = torch.clamp(std_val, min=1e-5)
    
    freq_magnitude_norm = (freq_magnitude_norm - mean_val) / std_val
    freq_magnitude_norm = torch.clamp(freq_magnitude_norm, -10, 10)
    freq_magnitude_norm = freq_magnitude_norm.view(b, c, h, w)
    
    freq_magnitude_norm = torch.nan_to_num(freq_magnitude_norm, nan=0.0, posinf=10.0, neginf=-10.0)
    
    return freq_magnitude_norm

# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(Dataset):
    """Custom dataset for frequency domain representations with caching"""
    
    def __init__(self, original_dataset, cache_freq=False):
        self.original_dataset = original_dataset
        self.cache_freq = cache_freq
        self.freq_cache = {} if cache_freq else None
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        if self.cache_freq and idx in self.freq_cache:
            return self.freq_cache[idx]
        
        image, label = self.original_dataset[idx]
        
        with torch.no_grad():
            freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
            freq_features = freq_features.squeeze(0)
            freq_phase = freq_phase.squeeze(0)
        
        result = (freq_features, label, freq_phase)
        
        if self.cache_freq:
            self.freq_cache[idx] = result
        
        return result

# ================== Step 4: EfficientNet CNN with Dual-Domain Feature Fusion ==================

class DualDomainFeatureFusion(nn.Module):
    """
    Dual-Domain Feature Fusion Module
    Combines spatial activation maps with their frequency domain representations
    """
    def __init__(self, in_channels):
        super(DualDomainFeatureFusion, self).__init__()
        
        self.spatial_weight = nn.Parameter(torch.tensor(0.7))
        self.freq_weight = nn.Parameter(torch.tensor(0.3))
        
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.Sigmoid()
        )
        
        self.bn_spatial = nn.BatchNorm2d(in_channels)
        self.bn_freq = nn.BatchNorm2d(in_channels)
        
    def forward(self, spatial_features):
        """
        Args:
            spatial_features: Activation maps from last conv layer (B, C, H, W)
        Returns:
            fused_features: Combined spatial + frequency features (B, C, H, W)
        """
        freq_features = extract_frequency_features(spatial_features)
        
        spatial_features_norm = self.bn_spatial(spatial_features)
        freq_features_norm = self.bn_freq(freq_features)
        
        combined = torch.cat([spatial_features_norm, freq_features_norm], dim=1)
        
        attention = self.channel_attention(combined)
        combined_weighted = combined * attention
        
        spatial_weighted = combined_weighted[:, :spatial_features.size(1), :, :]
        freq_weighted = combined_weighted[:, spatial_features.size(1):, :, :]
        
        spatial_w = torch.sigmoid(self.spatial_weight)
        freq_w = torch.sigmoid(self.freq_weight)
        
        fused = spatial_w * spatial_weighted + freq_w * freq_weighted
        fused = torch.nan_to_num(fused, nan=0.0, posinf=10.0, neginf=-10.0)
        
        return fused


class EnhancedFrequencyDomainCNN(nn.Module):
    """
    EfficientNet-B0 based model with Dual-Domain Feature Fusion
    Innovation: Second-level FFT on last conv layer before GAP
    """
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(EnhancedFrequencyDomainCNN, self).__init__()
        
        # Load pretrained EfficientNet-B0
        self.efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        
        # --- Modify first conv layer for 9-channel input ---
        # EfficientNet-B0 first conv: features[0][0] is a Conv2d(3, 32, 3, stride=2, padding=1)
        original_conv = self.efficientnet.features[0][0]
        new_conv = nn.Conv2d(
            9, 
            original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=(original_conv.bias is not None)
        )
        
        with torch.no_grad():
            # Copy weights for first 3 channels from pretrained
            new_conv.weight[:, :3, :, :] = original_conv.weight
            # Initialize remaining 6 channels with scaled Kaiming init
            nn.init.kaiming_normal_(new_conv.weight[:, 3:, :, :], mode='fan_out', nonlinearity='relu')
            new_conv.weight[:, 3:, :, :] *= 0.1  # Scale down frequency channel weights
            if original_conv.bias is not None and new_conv.bias is not None:
                new_conv.bias.copy_(original_conv.bias)
        
        self.efficientnet.features[0][0] = new_conv
        
        # --- Determine the output channels of the last feature layer ---
        # EfficientNet-B0 last conv block output is 1280 channels
        num_features_last = 1280
        
        # Add Dual-Domain Feature Fusion before GAP
        self.dual_domain_fusion = DualDomainFeatureFusion(num_features_last)
        
        # Replace the classifier
        self.efficientnet.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features_last, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        # Adaptive average pool (EfficientNet uses this by default, but we keep explicit)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self._initialize_new_weights()
    
    def _initialize_new_weights(self):
        """Initialize new layers (classifier and fusion) with proper weights"""
        for m in self.efficientnet.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
        for m in self.dual_domain_fusion.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def _forward_features(self, x):
        """Forward pass through EfficientNet feature extractor only"""
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        x = self.efficientnet.features(x)
        return x
    
    def forward(self, x):
        # Input validation
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        
        # Forward through EfficientNet feature extractor
        x = self.efficientnet.features(x)
        
        # Apply Dual-Domain Feature Fusion (x + fft(x))
        x = self.dual_domain_fusion(x)
        
        # Global Average Pooling
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Classifier
        x = self.efficientnet.classifier(x)
        
        return x
    
    def get_activations(self, x):
        """Extract feature maps from last conv layer before fusion"""
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        x = self.efficientnet.features(x)
        return x
    
    def get_fused_activations(self, x):
        """Extract feature maps after dual-domain fusion"""
        x = self.get_activations(x)
        x = self.dual_domain_fusion(x)
        return x

# ================== Step 5: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the enhanced frequency domain CNN model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if ('efficientnet.classifier' in name or 
            'efficientnet.features.0.0' in name or  # Modified first conv
            'dual_domain_fusion' in name):
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},
        {'params': new_params, 'lr': lr * 0.5}
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images = freq_images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            if torch.isnan(freq_images).any() or torch.isinf(freq_images).any():
                print(f"Warning: NaN/Inf detected in input batch {i}, skipping...")
                continue
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
            
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = running_loss / max(len(train_loader), 1)
        train_accuracy = 100 * correct_train / max(total_train, 1)
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for freq_images, labels, _ in val_pbar:
                freq_images = freq_images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                if scaler is not None:
                    with torch.amp.autocast('cuda'):
                        outputs = model(freq_images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(freq_images)
                    loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct / total:.2f}%'
                })
        
        avg_val_loss = running_val_loss / max(len(val_loader), 1)
        val_accuracy = 100 * correct / max(total, 1)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: Enhanced Score-CAM Implementation ==================

class ScoreCAM:
    """Score-CAM implementation with dual-domain awareness"""
    
    def __init__(self, model):
        self.model = model
        self.model.eval()
        
    def generate_cam(self, input_image, target_class, batch_size=16):
        """Generate Score-CAM using fused activations"""
        activations = self.model.get_fused_activations(input_image)
        b, k, h, w = activations.shape
        
        with torch.no_grad():
            base_output = self.model(input_image)
            base_score = F.softmax(base_output, dim=1)[0, target_class].item()
        
        _, _, input_h, input_w = input_image.shape
        upsampled_activations = F.interpolate(
            activations, 
            size=(input_h, input_w), 
            mode='bilinear', 
            align_corners=False
        )
        
        upsampled_activations = upsampled_activations.squeeze(0)
        
        weights = []
        
        # Process in batches to save memory (especially important for 1280 channels)
        for i in range(0, k, batch_size):
            batch_end = min(i + batch_size, k)
            batch_activations = upsampled_activations[i:batch_end]
            
            batch_weights = []
            for act_map in batch_activations:
                act_map_norm = act_map - act_map.min()
                if act_map_norm.max() > 0:
                    act_map_norm = act_map_norm / act_map_norm.max()
                
                masked_input = input_image * act_map_norm.unsqueeze(0).unsqueeze(0)
                
                with torch.no_grad():
                    output = self.model(masked_input)
                    score = F.softmax(output, dim=1)[0, target_class].item()
                
                batch_weights.append(score)
            
            weights.extend(batch_weights)
            
            # Free memory periodically
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        weights = torch.FloatTensor(weights).to(device)
        
        if weights.max() > 0:
            weights = weights / weights.max()
        
        activations_2d = activations.squeeze(0)
        cam = torch.zeros((h, w), dtype=torch.float32).to(device)
        
        for i, w_val in enumerate(weights):
            cam += w_val * activations_2d[i]
        
        cam = F.relu(cam)
        
        if cam.max() > 0:
            cam = cam / cam.max()
        
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=(input_h, input_w),
            mode='bilinear',
            align_corners=False
        ).squeeze()
        
        return cam.cpu().detach().numpy(), weights.cpu().detach().numpy()

# ================== Step 7: Spatial Domain Mapping ==================

def apply_scorecam_and_map_to_spatial(model, freq_image, phase, target_class, original_image):
    """Apply Score-CAM with dual-domain features"""
    
    model.eval()
    freq_input = freq_image.clone().detach().to(device)
    
    scorecam = ScoreCAM(model)
    cam_freq, weights = scorecam.generate_cam(freq_input, target_class, batch_size=32)
    
    freq_magnitude = freq_input[:, :3, :, :].squeeze(0).cpu().detach()
    
    cam_freq_tensor = torch.from_numpy(cam_freq).float()
    masked_freq_magnitude = freq_magnitude * cam_freq_tensor.unsqueeze(0)
    
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    elif phase.dim() == 2:
        phase = phase.unsqueeze(0).repeat(3, 1, 1)
    
    cam_spatial = frequency_to_spatial(
        masked_freq_magnitude.unsqueeze(0), 
        phase.unsqueeze(0)
    )
    cam_spatial = cam_spatial.squeeze(0)
    
    cam_spatial = torch.abs(cam_spatial)
    saliency_map = torch.mean(cam_spatial, dim=0).numpy()
    
    saliency_map = np.max(saliency_map) - saliency_map
    saliency_map = gaussian_filter(saliency_map, sigma=2.5)
    
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    
    threshold = np.percentile(saliency_map, 40)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)
    
    from scipy.ndimage import binary_closing, binary_opening
    binary_mask = saliency_map > 0
    binary_mask = binary_closing(binary_mask, structure=np.ones((5, 5)))
    binary_mask = binary_opening(binary_mask, structure=np.ones((3, 3)))
    
    saliency_map = saliency_map * binary_mask
    
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    
    saliency_map = np.power(saliency_map, 0.7)
    
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()
    
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha = 0.7 * saliency_map[:, :, np.newaxis]
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)
    
    return cam_spatial, saliency_map, highlighted, original_np, cam_freq

# ================== Step 8: Visualization Functions ==================

def plot_scorecam_results(original_np, freq_magnitude, cam_freq, saliency_map, 
                          highlighted, prediction, true_label, classes, confidence):
    """Plot comprehensive Score-CAM results with dual-domain info"""
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle('Enhanced Frequency Domain CNN (EfficientNet-B0) with Dual-Domain Feature Fusion\n'
                 '(Spatial + FFT of Last Conv Layer)', 
                 fontsize=16, fontweight='bold', y=0.998)
    
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}', 
                         fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')
    
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Input Magnitude Spectrum)', 
                         fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    im2 = axes[0, 2].imshow(cam_freq, cmap='jet')
    axes[0, 2].set_title('Score-CAM\n(Dual-Domain Features)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)
    
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 3].text(0.5, 0.5, f'{correct} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%', 
                    ha='center', va='center', fontsize=13, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 3].set_title('Model Prediction', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')
    
    im3 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Enhanced Features)', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Regions\n(Dual-Domain Aware)', 
                         fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50% Overlay)', fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')
    
    axes[1, 3].imshow(np.concatenate([original_np, highlighted], axis=1))
    axes[1, 3].set_title('Before | After\n(Enhanced Score-CAM)', 
                         fontsize=11, fontweight='bold')
    axes[1, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 9: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Enhanced Frequency Domain CNN (EfficientNet-B0) with Dual-Domain Feature Fusion")
    print("Innovation: x_fused = x + fft(x) before GAP layer")
    print("OPTIMIZED FOR 16GB GPU")
    print("="*80)
    
    # Dataset path - MODIFY THIS PATH
    data_root = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'
    
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please modify the 'data_root' variable to point to your dataset location.")
        return
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        print(f"Number of classes: {len(classes)}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    print("\n[Step 3] Converting to frequency domain using FFT...")
    freq_train_dataset = FrequencyDomainDataset(train_subset, cache_freq=False)
    freq_val_dataset = FrequencyDomainDataset(val_subset, cache_freq=False)
    freq_test_dataset = FrequencyDomainDataset(testset, cache_freq=False)
    
    # EfficientNet-B0 is lighter than ResNet50, so batch size can be larger
    batch_size = 128
    num_workers = 4 if os.name != 'nt' else 0
    
    print(f"Batch size: {batch_size}")
    print(f"Num workers: {num_workers}")
    
    train_loader = DataLoader(
        freq_train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        freq_val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        freq_test_dataset, 
        batch_size=1, 
        shuffle=False,
        num_workers=0
    )
    
    print("\n[Step 4] Initializing EfficientNet-B0 with Dual-Domain Fusion...")
    model = EnhancedFrequencyDomainCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("Architecture: EfficientNet-B0 + Dual-Domain Feature Fusion (x + fft(x)) before GAP")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
    print("\n[Step 5] Training model with dual-domain features...")
    print("Starting training loop...\n")
    
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader, 
            epochs=50,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for freq_images, labels, _ in test_pbar:
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            test_pbar.set_postfix({'acc': f'{100 * correct / total:.2f}%'})
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    print("\n[Step 7] Applying Score-CAM with dual-domain features...")
    print("Generating enhanced saliency maps using fused activation maps...\n")
    
    np.random.seed(42)
    num_samples = min(5, len(testset))
    test_indices = np.random.choice(len(testset), num_samples, replace=False)
    
    for idx in test_indices:
        try:
            original_image, true_label = testset[idx]
            
            freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
            freq_input = freq_features.to(device)
            
            model.eval()
            with torch.no_grad():
                output = model(freq_input)
                probabilities = F.softmax(output, dim=1)
                confidence, predicted = torch.max(probabilities.data, 1)
                predicted_class = predicted.item()
                confidence = confidence.item() * 100
            
            print(f"Sample {idx}:")
            print(f"  True Label: {classes[true_label]}")
            print(f"  Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
            
            _, saliency_map, highlighted, original_np, cam_freq = apply_scorecam_and_map_to_spatial(
                model, freq_input, phase.squeeze(0), predicted_class, original_image
            )
            
            plot_scorecam_results(
                original_np,
                freq_features, 
                cam_freq,
                saliency_map, 
                highlighted,
                predicted_class, 
                true_label, 
                classes,
                confidence
            )
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            print()
        
        except Exception as e:
            print(f"ERROR processing sample {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print("="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print(f"Total Classes: {len(classes)}")
    print("="*80)
    print("\n🚀 KEY INNOVATION:")
    print("✓ EfficientNet-B0 backbone (lighter & more efficient than ResNet50)")
    print("✓ Dual-Domain Feature Fusion: x_fused = α*x + β*fft(x)")
    print("✓ Applied before GAP layer to capture hidden frequency patterns")
    print("✓ Learnable fusion weights for optimal balance")
    print("✓ Channel attention for adaptive feature weighting")
    print("="*80)
    print("\nOptimizations Applied:")
    print("✓ Mixed precision training (AMP)")
    print("✓ Optimized batch size (128) for 16GB GPU")
    print("✓ Gradient clipping")
    print("✓ Memory-efficient data loading")
    print("✓ Periodic GPU cache clearing")
    print("✓ CuDNN benchmark mode enabled")
    print("="*80)
    
    print("\n[Step 8] Saving trained model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'test_accuracy': test_accuracy,
            'classes': classes,
            'num_classes': len(classes),
            'backbone': 'EfficientNet-B0'
        }, 'fruits_efficientnet_dual_domain_cnn.pth')
        print("Model saved as 'fruits_efficientnet_dual_domain_cnn.pth'")
    except Exception as e:
        print(f"ERROR saving model: {e}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\nFinal GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

if __name__ == "__main__":
    main()

In [ ]:
# Evaluation Script: Generate Overall Metrics Report from Saved Model
# Uses the saved .pth file to evaluate on the Fruits-360 test set

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, confusion_matrix
)
import os
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ================== Dataset Class ==================

class FruitsDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir)
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name),
                                           self.class_to_idx[class_name]))
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

# ================== FFT Conversion ==================

def spatial_to_frequency(images):
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    mean_val = freq_magnitude_log.mean()
    std_val = freq_magnitude_log.std()
    std_val = torch.clamp(std_val, min=1e-5)
    freq_magnitude_normalized = (freq_magnitude_log - mean_val) / std_val
    freq_magnitude_normalized = torch.clamp(freq_magnitude_normalized, -10, 10)
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    freq_features = torch.nan_to_num(freq_features, nan=0.0, posinf=10.0, neginf=-10.0)
    return freq_features, freq_phase, freq_complex

def extract_frequency_features(activation_map):
    freq_complex = fft.fft2(activation_map, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    freq_magnitude = torch.abs(freq_complex)
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    b, c, h, w = freq_magnitude_log.shape
    freq_magnitude_norm = freq_magnitude_log.view(b, c, -1)
    mean_val = freq_magnitude_norm.mean(dim=2, keepdim=True)
    std_val = freq_magnitude_norm.std(dim=2, keepdim=True)
    std_val = torch.clamp(std_val, min=1e-5)
    freq_magnitude_norm = (freq_magnitude_norm - mean_val) / std_val
    freq_magnitude_norm = torch.clamp(freq_magnitude_norm, -10, 10)
    freq_magnitude_norm = freq_magnitude_norm.view(b, c, h, w)
    freq_magnitude_norm = torch.nan_to_num(freq_magnitude_norm, nan=0.0, posinf=10.0, neginf=-10.0)
    return freq_magnitude_norm

# ================== Frequency Domain Dataset ==================

class FrequencyDomainDataset(Dataset):
    def __init__(self, original_dataset, cache_freq=False):
        self.original_dataset = original_dataset
        self.cache_freq = cache_freq
        self.freq_cache = {} if cache_freq else None

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        if self.cache_freq and idx in self.freq_cache:
            return self.freq_cache[idx]
        image, label = self.original_dataset[idx]
        with torch.no_grad():
            freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
            freq_features = freq_features.squeeze(0)
            freq_phase = freq_phase.squeeze(0)
        result = (freq_features, label, freq_phase)
        if self.cache_freq:
            self.freq_cache[idx] = result
        return result

# ================== Model Architecture (must match training) ==================

class DualDomainFeatureFusion(nn.Module):
    def __init__(self, in_channels):
        super(DualDomainFeatureFusion, self).__init__()
        self.spatial_weight = nn.Parameter(torch.tensor(0.7))
        self.freq_weight = nn.Parameter(torch.tensor(0.3))
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.Sigmoid()
        )
        self.bn_spatial = nn.BatchNorm2d(in_channels)
        self.bn_freq = nn.BatchNorm2d(in_channels)

    def forward(self, spatial_features):
        freq_features = extract_frequency_features(spatial_features)
        spatial_features_norm = self.bn_spatial(spatial_features)
        freq_features_norm = self.bn_freq(freq_features)
        combined = torch.cat([spatial_features_norm, freq_features_norm], dim=1)
        attention = self.channel_attention(combined)
        combined_weighted = combined * attention
        spatial_weighted = combined_weighted[:, :spatial_features.size(1), :, :]
        freq_weighted = combined_weighted[:, spatial_features.size(1):, :, :]
        spatial_w = torch.sigmoid(self.spatial_weight)
        freq_w = torch.sigmoid(self.freq_weight)
        fused = spatial_w * spatial_weighted + freq_w * freq_weighted
        fused = torch.nan_to_num(fused, nan=0.0, posinf=10.0, neginf=-10.0)
        return fused


class EnhancedFrequencyDomainCNN(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super(EnhancedFrequencyDomainCNN, self).__init__()
        self.efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        original_conv = self.efficientnet.features[0][0]
        new_conv = nn.Conv2d(
            9,
            original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=(original_conv.bias is not None)
        )
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = original_conv.weight
            nn.init.kaiming_normal_(new_conv.weight[:, 3:, :, :], mode='fan_out', nonlinearity='relu')
            new_conv.weight[:, 3:, :, :] *= 0.1
            if original_conv.bias is not None and new_conv.bias is not None:
                new_conv.bias.copy_(original_conv.bias)
        self.efficientnet.features[0][0] = new_conv
        num_features_last = 1280
        self.dual_domain_fusion = DualDomainFeatureFusion(num_features_last)
        self.efficientnet.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features_last, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        x = self.efficientnet.features(x)
        x = self.dual_domain_fusion(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.efficientnet.classifier(x)
        return x

# ================== Evaluation Function ==================

def evaluate_and_generate_report(model_path, data_root, output_txt_path):
    """Load saved model, evaluate on test set, and generate metrics report."""

    # --- Load checkpoint ---
    print(f"\nLoading checkpoint from: {model_path}")
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    classes = checkpoint['classes']
    num_classes = checkpoint['num_classes']
    backbone = checkpoint.get('backbone', 'EfficientNet-B0')
    print(f"Backbone: {backbone}")
    print(f"Number of classes: {num_classes}")
    print(f"Saved test accuracy: {checkpoint.get('test_accuracy', 'N/A')}")

    # --- Build model and load weights ---
    print("\nBuilding model architecture...")
    model = EnhancedFrequencyDomainCNN(num_classes=num_classes, dropout_rate=0.5).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print("Model weights loaded successfully.")

    # --- Prepare test dataset ---
    print("\nPreparing test dataset...")
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_dir = os.path.join(data_root, 'Test')
    testset = FruitsDataset(test_dir, transform=transform_test)
    freq_test_dataset = FrequencyDomainDataset(testset, cache_freq=False)

    batch_size = 128
    num_workers = 4 if os.name != 'nt' else 0
    test_loader = DataLoader(
        freq_test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    # --- Run inference ---
    print("\nRunning inference on test set...")
    all_predictions = []
    all_labels = []

    scaler_available = torch.cuda.is_available()

    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Evaluating"):
            freq_images = freq_images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if scaler_available:
                with torch.amp.autocast('cuda'):
                    outputs = model(freq_images)
            else:
                outputs = model(freq_images)

            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    total_samples = len(all_labels)
    correctly_classified = int(np.sum(all_predictions == all_labels))
    misclassified = total_samples - correctly_classified

    # --- Compute overall metrics ---
    print("\nComputing metrics...")

    overall_accuracy = accuracy_score(all_labels, all_predictions)
    overall_precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_kappa = cohen_kappa_score(all_labels, all_predictions)

    # Sensitivity = Recall (macro-averaged)
    overall_sensitivity = overall_recall

    # Specificity: macro-averaged across all classes
    cm = confusion_matrix(all_labels, all_predictions, labels=list(range(num_classes)))
    specificities = []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp
        if (tn + fp) > 0:
            specificities.append(tn / (tn + fp))
        else:
            specificities.append(0.0)
    overall_specificity = np.mean(specificities)

    # Error rate (macro-averaged): 1 - recall per class, then average
    recalls_per_class = recall_score(all_labels, all_predictions, average=None, zero_division=0)
    error_rates_per_class = 1.0 - recalls_per_class
    overall_error_rate = np.mean(error_rates_per_class)

    # --- Print metrics ---
    print(f"\nOverall Accuracy:      {overall_accuracy:.6f} ({overall_accuracy*100:.4f}%)")
    print(f"Overall Precision:     {overall_precision:.6f}")
    print(f"Overall Recall:        {overall_recall:.6f}")
    print(f"Overall F1-Score:      {overall_f1:.6f}")
    print(f"Overall Cohen's Kappa: {overall_kappa:.6f}")
    print(f"Overall Sensitivity:   {overall_sensitivity:.6f}")
    print(f"Overall Specificity:   {overall_specificity:.6f}")
    print(f"Overall Error Rate:    {overall_error_rate:.6f}")

    # --- Generate report text file ---
    print(f"\nGenerating report: {output_txt_path}")

    report_lines = []
    sep = "=" * 100
    dash = "-" * 100

    report_lines.append(sep)
    report_lines.append("OVERALL EVALUATION METRICS REPORT")
    report_lines.append("Enhanced Dual Domain CNN with Feature Fusion - Fruits-360 Dataset")
    report_lines.append(f"Model Architecture: {backbone} + Dual-Domain Feature Fusion (Spatial + FFT)")
    report_lines.append("Input: 9 channels (3 Magnitude + 3 Phase Cosine + 3 Phase Sine)")
    report_lines.append(sep)
    report_lines.append("")
    report_lines.append("OVERALL PERFORMANCE METRICS:")
    report_lines.append(dash)
    report_lines.append(f"Overall Accuracy:           {overall_accuracy:.6f} ({overall_accuracy*100:.4f}%)")
    report_lines.append(f"Overall Cohen's Kappa:      {overall_kappa:.6f}")
    report_lines.append(f"Overall Precision:          {overall_precision:.6f}")
    report_lines.append(f"Overall Recall:             {overall_recall:.6f}")
    report_lines.append(f"Overall F1-Score:           {overall_f1:.6f}")
    report_lines.append(f"Overall Sensitivity:        {overall_sensitivity:.6f}")
    report_lines.append(f"Overall Specificity:        {overall_specificity:.6f}")
    report_lines.append(f"Overall Error Rate:         {overall_error_rate:.6f}")
    report_lines.append(dash)
    report_lines.append("")
    report_lines.append("DATASET INFORMATION:")
    report_lines.append(dash)
    report_lines.append(f"Total Test Samples:         {total_samples}")
    report_lines.append(f"Number of Classes:          {num_classes}")
    report_lines.append(f"Correctly Classified:       {correctly_classified}")
    report_lines.append(f"Misclassified:              {misclassified}")
    report_lines.append(dash)
    report_lines.append("")
    report_lines.append("METRIC DEFINITIONS:")
    report_lines.append(dash)
    report_lines.append("Accuracy:      Ratio of correctly predicted samples to total samples")
    report_lines.append("Cohen's Kappa: Statistical measure of inter-rater agreement (accounts for chance)")
    report_lines.append("Precision:     Ratio of true positives to all predicted positives (macro-averaged)")
    report_lines.append("Recall:        Ratio of true positives to all actual positives (macro-averaged)")
    report_lines.append("F1-Score:      Harmonic mean of precision and recall (macro-averaged)")
    report_lines.append("Sensitivity:   True positive rate (same as recall, macro-averaged)")
    report_lines.append("Specificity:   True negative rate (macro-averaged)")
    report_lines.append("Error Rate:    Ratio of misclassified samples (macro-averaged)")
    report_lines.append(dash)
    report_lines.append("")
    report_lines.append(sep)
    report_lines.append("END OF OVERALL METRICS REPORT")
    report_lines.append(sep)

    report_text = "\n".join(report_lines)

    with open(output_txt_path, 'w') as f:
        f.write(report_text)

    print(f"\nReport saved to: {output_txt_path}")
    print("\n" + report_text)

    return {
        'accuracy': overall_accuracy,
        'precision': overall_precision,
        'recall': overall_recall,
        'f1': overall_f1,
        'kappa': overall_kappa,
        'sensitivity': overall_sensitivity,
        'specificity': overall_specificity,
        'error_rate': overall_error_rate
    }


# ================== Main ==================

if __name__ == "__main__":
    # === MODIFY THESE PATHS AS NEEDED ===
    MODEL_PATH = r'fruits_efficientnet_dual_domain_cnn.pth'
    DATA_ROOT = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'
    OUTPUT_TXT = r'overall_evaluation_metrics_report.txt'

    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: Model file not found: {MODEL_PATH}")
        print("Please update MODEL_PATH to the correct location of your .pth file.")
    elif not os.path.exists(DATA_ROOT):
        print(f"ERROR: Dataset path not found: {DATA_ROOT}")
        print("Please update DATA_ROOT to the correct location of Fruits-360 dataset.")
    else:
        metrics = evaluate_and_generate_report(MODEL_PATH, DATA_ROOT, OUTPUT_TXT)
        print("\nDone!")